In [1]:
import numpy as np
import pandas as pd
import argparse
import os
from itertools import combinations
from collections import defaultdict
from upsetplot import from_memberships, UpSet
from upsetplot import plot
import matplotlib.pyplot as plt
import logging
import csv

In [2]:
# initialize dataFrames
def initialize_dfs(names):
    return {name: pd.DataFrame() for name in names}
    
# function to read csv files into dataframes
def read_csv_files(names, df_dict, base_path, columns):
    for name in names:
        file_path = os.path.join(base_path, name, "produce_final_form/final_form.csv")
        df_dict[name] = pd.read_csv(file_path, sep=',', usecols=columns)

def find_common_sequences(samples):
    combos = []
    counts = []

    # Iterate through all combinations of sample keys (categories)
    for i in range(1, len(samples) + 1):
        for combo in combinations(samples.keys(), i):
            # Find common sequences and their count
            common_seqs = set.intersection(*(set(samples[sample]) for sample in combo))
            combos.append(list(combo))  # Store the combination as a list
            counts.append(len(common_seqs))  # Store the count of common sequences

    return combos, counts

# Generate the final list format
def format_memberships(sample_sequences, result):
    formatted = []
    data = []
    
# Filter conditions: mirBase, linearfold, mirdeep2
def filter_dataframes(names, df_source, df_with_match):
    for name in names:
        df_with_match[name] = filter_any_condition_met(df_source[name])
        # df_all_zero[name] = filter_none_met(df_source[name])

def filter_any_condition_met(df):
    filtered_df = df[(df[['mirBase', 'linearfold', 'mirdeep2']].sum(axis=1) > 0)]
    filtered_df = filtered_df.loc[:, ['sequence', 'qseqid_count']]
    # print("filtered_df",filtered_df)
    return filtered_df

def shared_by_at_least_two(all_sequences):
    all_sets = {k: set(v) for k, v in all_sequences.items()}

    total_set = set.union(*all_sets.values())  # union
    unique_only = set()
    seen = set()

    print(f"Initial: seen={seen}, unique_only={unique_only}\n")

    for name, s in all_sets.items():
        print(f"After {name}:")
        duplicates = seen & s
        unique_only |= (s - seen)
        unique_only -= duplicates
        seen |= s
        # print(f"After {name}:")
        # print(f"  sample={s}")
        # print(f"  duplicates={duplicates}")
        # print(f"  seen={seen}")
        # print(f"  unique_only={unique_only}\n")

    shared = total_set - unique_only
    return len(total_set), len(unique_only), len(shared)

In [3]:

# define the base path for output csv files
base_path = "../../../../1st_11samples_output"
#base_path = "../../../../3rd_3samples_output"
#base_path = "../../../../4th_3samples_output"
#base_path = "../../../../6th_2samples_output"
#base_path = "../../../../7th_6samples_output"
# base_path = "../../../../8th_2samples_output"
# base_path = "../../../../10th_2samples_output"
#base_path = "../../../../11th_3samples_output"
#base_path = "../../../../13th_3samples_output"

# names corresponding to the file names for bifi and ecoli
# sample_names = [
#     "MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001", "MEE-BIEVs-3_S9_L001_R1_001", 
#     "MEF-BIEVs-1_S1_L001_R1_001", "MEF-BIEVs-2_S5_L001_R1_001", "MEF-BIEVs-3_S6_L001_R1_001"
# ]
#dataset2
sample_names = [
    "SRR18745680", "SRR18745681", "SRR18745682", 
    "SRR18745683", "SRR18745684", "SRR18745685",
    "SRR18745686", "SRR18745687", "SRR18745688", 
    "SRR18745689", "SRR18745690"
]
#dataset3
# sample_names = [
#     "SRR20723831", "SRR20723832", "SRR20723833"
# ]
#dataset4
# sample_names = [
#     "SRR25896993", "SRR25896994", "SRR25896995"
# ]
#dataset6
# sample_names = [
#     "SRR12412943", "SRR12412944"
# ]
#dataset7
# sample_names = [
#     "DRR037104", "DRR037105","DRR037108","DRR037109","DRR037112","DRR037113"
# ]
# #dataset8
# sample_names = [
#     "SRR1638859", "SRR1638860"
# ]
#dataset10
# sample_names = [
#     "SRR943111", "SRR943112"
# ]
# #dataset11
# sample_names = [
#     "SRR684065", "SRR684066","SRR684086"
# ]
#dataset13
# sample_names = [
#     "SRR8760721", "SRR8760722","SRR8760723"
# ]
# # bifi_names = ["MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001", "MEE-BIEVs-3_S9_L001_R1_001"]
# ecoli_names = [
#     "MEE-OMVs-1_S4_L001_R1_001", "MEE-OMVs-2_S3_L001_R1_001", "MEE-OMVs-3_S12_L001_R1_001", 
#     "MEF-OMVs-1_S2_L001_R1_001", "MEF-OMVs-2_S11_L001_R1_001", "MEF-OMVs-3_S10_L001_R1_001"
# ]

# columns to read from the original CSV files
read_columns = ['sequence', 'mirBase', 'linearfold', 'mirdeep2','qseqid_count']

df_all = initialize_dfs(sample_names)
df_all_with_match = initialize_dfs(sample_names)
# df_ecoli = initialize_dfs(ecoli_names)
# df_all = initialize_dfs(bifi_names + ecoli_names)

#df_bifi is a dictionary
read_csv_files(sample_names, df_all, base_path, read_columns)
filter_dataframes(sample_names, df_all, df_all_with_match)
total_num_rows = 0
for name in sample_names:
    num_rows = len(df_all_with_match[name])
    print(f"DataFrame {name} has {num_rows} lines.")
    total_num_rows += num_rows
print("total", total_num_rows)
all_sequences = {name: list(seq for seq in df['sequence']) for name, df in df_all_with_match.items()}
print("1")


total, unique_only_cnt, shared_cnt = shared_by_at_least_two(all_sequences)
print("RESULTS")

print("Total unique sequences:", total)
print("Unique to one sample:  ", unique_only_cnt)
print("Shared by ≥2 samples:  ", shared_cnt)
print("Percentage shared:     ", f"{shared_cnt/total*100:.2f}%")

DataFrame SRR18745680 has 197930 lines.
DataFrame SRR18745681 has 392413 lines.
DataFrame SRR18745682 has 550261 lines.
DataFrame SRR18745683 has 180954 lines.
DataFrame SRR18745684 has 37644 lines.
DataFrame SRR18745685 has 46439 lines.
DataFrame SRR18745686 has 71943 lines.
DataFrame SRR18745687 has 48290 lines.
DataFrame SRR18745688 has 54957 lines.
DataFrame SRR18745689 has 63075 lines.
DataFrame SRR18745690 has 46615 lines.
total 1690521
1
Initial: seen=set(), unique_only=set()

After SRR18745680:
After SRR18745681:
After SRR18745682:
After SRR18745683:
After SRR18745684:
After SRR18745685:
After SRR18745686:
After SRR18745687:
After SRR18745688:
After SRR18745689:
After SRR18745690:
RESULTS
Total unique sequences: 1520705
Unique to one sample:   1398918
Shared by ≥2 samples:   121787
Percentage shared:      8.01%


In [3]:

# define the base path for output csv files
base_path = "../../../../2nd_6samples_output"
#base_path = "../../../../3rd_3samples_output"
#base_path = "../../../../4th_3samples_output"
#base_path = "../../../../6th_2samples_output"
#base_path = "../../../../7th_6samples_output"
# base_path = "../../../../8th_2samples_output"
# base_path = "../../../../10th_2samples_output"
#base_path = "../../../../11th_3samples_output"
#base_path = "../../../../13th_3samples_output"

# names corresponding to the file names for bifi and ecoli
# sample_names = [
#     "MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001", "MEE-BIEVs-3_S9_L001_R1_001", 
#     "MEF-BIEVs-1_S1_L001_R1_001", "MEF-BIEVs-2_S5_L001_R1_001", "MEF-BIEVs-3_S6_L001_R1_001"
# ]
#dataset2
sample_names = [
    "SRR18078867", "SRR18078868", "SRR18078869", 
    "SRR18078870", "SRR18078871", "SRR18078872"
]
#dataset3
# sample_names = [
#     "SRR20723831", "SRR20723832", "SRR20723833"
# ]
#dataset4
# sample_names = [
#     "SRR25896993", "SRR25896994", "SRR25896995"
# ]
#dataset6
# sample_names = [
#     "SRR12412943", "SRR12412944"
# ]
#dataset7
# sample_names = [
#     "DRR037104", "DRR037105","DRR037108","DRR037109","DRR037112","DRR037113"
# ]
# #dataset8
# sample_names = [
#     "SRR1638859", "SRR1638860"
# ]
#dataset10
# sample_names = [
#     "SRR943111", "SRR943112"
# ]
# #dataset11
# sample_names = [
#     "SRR684065", "SRR684066","SRR684086"
# ]
#dataset13
# sample_names = [
#     "SRR8760721", "SRR8760722","SRR8760723"
# ]
# # bifi_names = ["MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001", "MEE-BIEVs-3_S9_L001_R1_001"]
# ecoli_names = [
#     "MEE-OMVs-1_S4_L001_R1_001", "MEE-OMVs-2_S3_L001_R1_001", "MEE-OMVs-3_S12_L001_R1_001", 
#     "MEF-OMVs-1_S2_L001_R1_001", "MEF-OMVs-2_S11_L001_R1_001", "MEF-OMVs-3_S10_L001_R1_001"
# ]

# columns to read from the original CSV files
read_columns = ['sequence', 'mirBase', 'linearfold', 'mirdeep2','qseqid_count']

df_all = initialize_dfs(sample_names)
df_all_with_match = initialize_dfs(sample_names)
# df_ecoli = initialize_dfs(ecoli_names)
# df_all = initialize_dfs(bifi_names + ecoli_names)

#df_bifi is a dictionary
read_csv_files(sample_names, df_all, base_path, read_columns)
filter_dataframes(sample_names, df_all, df_all_with_match)
total_num_rows = 0
for name in sample_names:
    num_rows = len(df_all_with_match[name])
    print(f"DataFrame {name} has {num_rows} lines.")
    total_num_rows += num_rows
print("total", total_num_rows)
all_sequences = {name: list(seq for seq in df['sequence']) for name, df in df_all_with_match.items()}
print("1")


total, unique_only_cnt, shared_cnt = shared_by_at_least_two(all_sequences)
print("RESULTS")

print("Total unique sequences:", total)
print("Unique to one sample:  ", unique_only_cnt)
print("Shared by ≥2 samples:  ", shared_cnt)
print("Percentage shared:     ", f"{shared_cnt/total*100:.2f}%")


DataFrame SRR18078867 has 73721 lines.
DataFrame SRR18078868 has 49594 lines.
DataFrame SRR18078869 has 38450 lines.
DataFrame SRR18078870 has 72808 lines.
DataFrame SRR18078871 has 38747 lines.
DataFrame SRR18078872 has 50518 lines.
total 323838
1
Initial: seen=set(), unique_only=set()

After SRR18078867:
After SRR18078868:
After SRR18078869:
After SRR18078870:
After SRR18078871:
After SRR18078872:
RESULTS
Total unique sequences: 186577
Unique to one sample:   110334
Shared by ≥2 samples:   76243
Percentage shared:      40.86%


In [4]:

# define the base path for output csv files
#base_path = "../../../../2nd_6samples_output"
base_path = "../../../../3rd_3samples_output"
#base_path = "../../../../4th_3samples_output"
#base_path = "../../../../6th_2samples_output"
#base_path = "../../../../7th_6samples_output"
# base_path = "../../../../8th_2samples_output"
# base_path = "../../../../10th_2samples_output"
#base_path = "../../../../11th_3samples_output"
#base_path = "../../../../13th_3samples_output"

# names corresponding to the file names for bifi and ecoli
# sample_names = [
#     "MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001", "MEE-BIEVs-3_S9_L001_R1_001", 
#     "MEF-BIEVs-1_S1_L001_R1_001", "MEF-BIEVs-2_S5_L001_R1_001", "MEF-BIEVs-3_S6_L001_R1_001"
# ]
#dataset2
# sample_names = [
#     "SRR18078867", "SRR18078868", "SRR18078869", 
#     "SRR18078870", "SRR18078871", "SRR18078872"
# ]
#dataset3
sample_names = [
    "SRR20723831", "SRR20723832", "SRR20723833"
]
#dataset4
# sample_names = [
#     "SRR25896993", "SRR25896994", "SRR25896995"
# ]
#dataset6
# sample_names = [
#     "SRR12412943", "SRR12412944"
# ]
#dataset7
# sample_names = [
#     "DRR037104", "DRR037105","DRR037108","DRR037109","DRR037112","DRR037113"
# ]
# #dataset8
# sample_names = [
#     "SRR1638859", "SRR1638860"
# ]
#dataset10
# sample_names = [
#     "SRR943111", "SRR943112"
# ]
# #dataset11
# sample_names = [
#     "SRR684065", "SRR684066","SRR684086"
# ]
#dataset13
# sample_names = [
#     "SRR8760721", "SRR8760722","SRR8760723"
# ]
# # bifi_names = ["MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001", "MEE-BIEVs-3_S9_L001_R1_001"]
# ecoli_names = [
#     "MEE-OMVs-1_S4_L001_R1_001", "MEE-OMVs-2_S3_L001_R1_001", "MEE-OMVs-3_S12_L001_R1_001", 
#     "MEF-OMVs-1_S2_L001_R1_001", "MEF-OMVs-2_S11_L001_R1_001", "MEF-OMVs-3_S10_L001_R1_001"
# ]

# columns to read from the original CSV files
read_columns = ['sequence', 'mirBase', 'linearfold', 'mirdeep2','qseqid_count']

df_all = initialize_dfs(sample_names)
df_all_with_match = initialize_dfs(sample_names)
# df_ecoli = initialize_dfs(ecoli_names)
# df_all = initialize_dfs(bifi_names + ecoli_names)

#df_bifi is a dictionary
read_csv_files(sample_names, df_all, base_path, read_columns)
filter_dataframes(sample_names, df_all, df_all_with_match)
total_num_rows = 0
for name in sample_names:
    num_rows = len(df_all_with_match[name])
    print(f"DataFrame {name} has {num_rows} lines.")
    total_num_rows += num_rows
print("total", total_num_rows)
all_sequences = {name: list(seq for seq in df['sequence']) for name, df in df_all_with_match.items()}
print("1")


total, unique_only_cnt, shared_cnt = shared_by_at_least_two(all_sequences)
print("RESULTS")

print("Total unique sequences:", total)
print("Unique to one sample:  ", unique_only_cnt)
print("Shared by ≥2 samples:  ", shared_cnt)
print("Percentage shared:     ", f"{shared_cnt/total*100:.2f}%")


DataFrame SRR20723831 has 1873660 lines.
DataFrame SRR20723832 has 1793252 lines.
DataFrame SRR20723833 has 1952763 lines.
total 5619675
1
Initial: seen=set(), unique_only=set()

After SRR20723831:
After SRR20723832:
After SRR20723833:
RESULTS
Total unique sequences: 3602055
Unique to one sample:   2232258
Shared by ≥2 samples:   1369797
Percentage shared:      38.03%


In [5]:

# define the base path for output csv files
#base_path = "../../../../2nd_6samples_output"
#base_path = "../../../../3rd_3samples_output"
base_path = "../../../../4th_3samples_output"
#base_path = "../../../../6th_2samples_output"
#base_path = "../../../../7th_6samples_output"
# base_path = "../../../../8th_2samples_output"
# base_path = "../../../../10th_2samples_output"
#base_path = "../../../../11th_3samples_output"
#base_path = "../../../../13th_3samples_output"

# names corresponding to the file names for bifi and ecoli
# sample_names = [
#     "MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001", "MEE-BIEVs-3_S9_L001_R1_001", 
#     "MEF-BIEVs-1_S1_L001_R1_001", "MEF-BIEVs-2_S5_L001_R1_001", "MEF-BIEVs-3_S6_L001_R1_001"
# ]
#dataset2
# sample_names = [
#     "SRR18078867", "SRR18078868", "SRR18078869", 
#     "SRR18078870", "SRR18078871", "SRR18078872"
# ]
#dataset3
# sample_names = [
#     "SRR20723831", "SRR20723832", "SRR20723833"
# ]
#dataset4
sample_names = [
    "SRR25896993", "SRR25896994", "SRR25896995"
]
#dataset6
# sample_names = [
#     "SRR12412943", "SRR12412944"
# ]
#dataset7
# sample_names = [
#     "DRR037104", "DRR037105","DRR037108","DRR037109","DRR037112","DRR037113"
# ]
# #dataset8
# sample_names = [
#     "SRR1638859", "SRR1638860"
# ]
#dataset10
# sample_names = [
#     "SRR943111", "SRR943112"
# ]
# #dataset11
# sample_names = [
#     "SRR684065", "SRR684066","SRR684086"
# ]
#dataset13
# sample_names = [
#     "SRR8760721", "SRR8760722","SRR8760723"
# ]
# # bifi_names = ["MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001", "MEE-BIEVs-3_S9_L001_R1_001"]
# ecoli_names = [
#     "MEE-OMVs-1_S4_L001_R1_001", "MEE-OMVs-2_S3_L001_R1_001", "MEE-OMVs-3_S12_L001_R1_001", 
#     "MEF-OMVs-1_S2_L001_R1_001", "MEF-OMVs-2_S11_L001_R1_001", "MEF-OMVs-3_S10_L001_R1_001"
# ]

# columns to read from the original CSV files
read_columns = ['sequence', 'mirBase', 'linearfold', 'mirdeep2','qseqid_count']

df_all = initialize_dfs(sample_names)
df_all_with_match = initialize_dfs(sample_names)
# df_ecoli = initialize_dfs(ecoli_names)
# df_all = initialize_dfs(bifi_names + ecoli_names)

#df_bifi is a dictionary
read_csv_files(sample_names, df_all, base_path, read_columns)
filter_dataframes(sample_names, df_all, df_all_with_match)
total_num_rows = 0
for name in sample_names:
    num_rows = len(df_all_with_match[name])
    print(f"DataFrame {name} has {num_rows} lines.")
    total_num_rows += num_rows
print("total", total_num_rows)
all_sequences = {name: list(seq for seq in df['sequence']) for name, df in df_all_with_match.items()}
print("1")


total, unique_only_cnt, shared_cnt = shared_by_at_least_two(all_sequences)
print("RESULTS")

print("Total unique sequences:", total)
print("Unique to one sample:  ", unique_only_cnt)
print("Shared by ≥2 samples:  ", shared_cnt)
print("Percentage shared:     ", f"{shared_cnt/total*100:.2f}%")


DataFrame SRR25896993 has 321739 lines.
DataFrame SRR25896994 has 322958 lines.
DataFrame SRR25896995 has 358302 lines.
total 1002999
1
Initial: seen=set(), unique_only=set()

After SRR25896993:
After SRR25896994:
After SRR25896995:
RESULTS
Total unique sequences: 748943
Unique to one sample:   577747
Shared by ≥2 samples:   171196
Percentage shared:      22.86%


In [6]:
# define the base path for output csv files
#base_path = "../../../../2nd_6samples_output"
#base_path = "../../../../3rd_3samples_output"
#base_path = "../../../../4th_3samples_output"
base_path = "../../../../5th_9samples_output"
#base_path = "../../../../6th_2samples_output"
#base_path = "../../../../7th_6samples_output"
# base_path = "../../../../8th_2samples_output"
# base_path = "../../../../10th_2samples_output"
#base_path = "../../../../11th_3samples_output"
#base_path = "../../../../13th_3samples_output"

# names corresponding to the file names for bifi and ecoli
# sample_names = [
#     "MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001", "MEE-BIEVs-3_S9_L001_R1_001", 
#     "MEF-BIEVs-1_S1_L001_R1_001", "MEF-BIEVs-2_S5_L001_R1_001", "MEF-BIEVs-3_S6_L001_R1_001"
# ]
#dataset2
# sample_names = [
#     "SRR18078867", "SRR18078868", "SRR18078869", 
#     "SRR18078870", "SRR18078871", "SRR18078872"
# ]
#dataset3
# sample_names = [
#     "SRR20723831", "SRR20723832", "SRR20723833"
# ]
#dataset4
# sample_names = [
#     "SRR25896993", "SRR25896994", "SRR25896995"
# ]
#dataset5
sample_names = [
                    "SRR8195524", "SRR8195525", "SRR8195526", "SRR8195527", "SRR8195528", "SRR8195529", "SRR8195530", "SRR8195531", "SRR8195532"
               ]
#dataset6
# sample_names = [
#     "SRR12412943", "SRR12412944"
# ]
#dataset7
# sample_names = [
#     "DRR037104", "DRR037105","DRR037108","DRR037109","DRR037112","DRR037113"
# ]
# #dataset8
# sample_names = [
#     "SRR1638859", "SRR1638860"
# ]
#dataset10
# sample_names = [
#     "SRR943111", "SRR943112"
# ]
# #dataset11
# sample_names = [
#     "SRR684065", "SRR684066","SRR684086"
# ]
#dataset13
# sample_names = [
#     "SRR8760721", "SRR8760722","SRR8760723"
# ]
# # bifi_names = ["MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001", "MEE-BIEVs-3_S9_L001_R1_001"]
# ecoli_names = [
#     "MEE-OMVs-1_S4_L001_R1_001", "MEE-OMVs-2_S3_L001_R1_001", "MEE-OMVs-3_S12_L001_R1_001", 
#     "MEF-OMVs-1_S2_L001_R1_001", "MEF-OMVs-2_S11_L001_R1_001", "MEF-OMVs-3_S10_L001_R1_001"
# ]

# columns to read from the original CSV files
read_columns = ['sequence', 'mirBase', 'linearfold', 'mirdeep2','qseqid_count']

df_all = initialize_dfs(sample_names)
df_all_with_match = initialize_dfs(sample_names)
# df_ecoli = initialize_dfs(ecoli_names)
# df_all = initialize_dfs(bifi_names + ecoli_names)

#df_bifi is a dictionary
read_csv_files(sample_names, df_all, base_path, read_columns)
filter_dataframes(sample_names, df_all, df_all_with_match)
total_num_rows = 0
for name in sample_names:
    num_rows = len(df_all_with_match[name])
    print(f"DataFrame {name} has {num_rows} lines.")
    total_num_rows += num_rows
print("total", total_num_rows)
all_sequences = {name: list(seq for seq in df['sequence']) for name, df in df_all_with_match.items()}
print("1")


total, unique_only_cnt, shared_cnt = shared_by_at_least_two(all_sequences)
print("RESULTS")

print("Total unique sequences:", total)
print("Unique to one sample:  ", unique_only_cnt)
print("Shared by ≥2 samples:  ", shared_cnt)
print("Percentage shared:     ", f"{shared_cnt/total*100:.2f}%")


DataFrame SRR8195524 has 907727 lines.
DataFrame SRR8195525 has 27591 lines.
DataFrame SRR8195526 has 818567 lines.
DataFrame SRR8195527 has 880830 lines.
DataFrame SRR8195528 has 434568 lines.
DataFrame SRR8195529 has 887957 lines.
DataFrame SRR8195530 has 1161864 lines.
DataFrame SRR8195531 has 1107085 lines.
DataFrame SRR8195532 has 877762 lines.
total 7103951
1
Initial: seen=set(), unique_only=set()

After SRR8195524:
After SRR8195525:
After SRR8195526:
After SRR8195527:
After SRR8195528:
After SRR8195529:
After SRR8195530:
After SRR8195531:
After SRR8195532:
RESULTS
Total unique sequences: 6718413
Unique to one sample:   6477881
Shared by ≥2 samples:   240532
Percentage shared:      3.58%


In [4]:
# define the base path for output csv files
#base_path = "../../../../2nd_6samples_output"
#base_path = "../../../../3rd_3samples_output"
#base_path = "../../../../4th_3samples_output"
base_path = "../../../../6th_8samples_output"
#base_path = "../../../../7th_6samples_output"
# base_path = "../../../../8th_2samples_output"
# base_path = "../../../../10th_2samples_output"
#base_path = "../../../../11th_3samples_output"
#base_path = "../../../../13th_3samples_output"

# names corresponding to the file names for bifi and ecoli
# sample_names = [
#     "MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001", "MEE-BIEVs-3_S9_L001_R1_001", 
#     "MEF-BIEVs-1_S1_L001_R1_001", "MEF-BIEVs-2_S5_L001_R1_001", "MEF-BIEVs-3_S6_L001_R1_001"
# ]
# #dataset2
# sample_names = [
#     "SRR18078867", "SRR18078868", "SRR18078869", 
#     "SRR18078870", "SRR18078871", "SRR18078872"
# ]
#dataset3
# sample_names = [
#     "SRR20723831", "SRR20723832", "SRR20723833"
# ]
#dataset4
# sample_names = [
#     "SRR25896993", "SRR25896994", "SRR25896995"
# ]
#dataset6
sample_names = ["ERR629132","ERR629133","ERR629134",
                "ERR629135","ERR629136","ERR629137",
                "ERR629138","ERR629139"
                    
                ]
#dataset7
# sample_names = [
#     "DRR037104", "DRR037105","DRR037108","DRR037109","DRR037112","DRR037113"
# ]
# #dataset8
# sample_names = [
#     "SRR1638859", "SRR1638860"
# ]
#dataset10
# sample_names = [
#     "SRR943111", "SRR943112"
# ]
# #dataset11
# sample_names = [
#     "SRR684065", "SRR684066","SRR684086"
# ]
#dataset13
# sample_names = [
#     "SRR8760721", "SRR8760722","SRR8760723"
# ]
# # bifi_names = ["MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001", "MEE-BIEVs-3_S9_L001_R1_001"]
# ecoli_names = [
#     "MEE-OMVs-1_S4_L001_R1_001", "MEE-OMVs-2_S3_L001_R1_001", "MEE-OMVs-3_S12_L001_R1_001", 
#     "MEF-OMVs-1_S2_L001_R1_001", "MEF-OMVs-2_S11_L001_R1_001", "MEF-OMVs-3_S10_L001_R1_001"
# ]

# columns to read from the original CSV files
read_columns = ['sequence', 'mirBase', 'linearfold', 'mirdeep2','qseqid_count']

df_all = initialize_dfs(sample_names)
df_all_with_match = initialize_dfs(sample_names)
# df_ecoli = initialize_dfs(ecoli_names)
# df_all = initialize_dfs(bifi_names + ecoli_names)

#df_bifi is a dictionary
read_csv_files(sample_names, df_all, base_path, read_columns)
filter_dataframes(sample_names, df_all, df_all_with_match)
total_num_rows = 0
for name in sample_names:
    num_rows = len(df_all_with_match[name])
    print(f"DataFrame {name} has {num_rows} lines.")
    total_num_rows += num_rows
print("total", total_num_rows)
all_sequences = {name: list(seq for seq in df['sequence']) for name, df in df_all_with_match.items()}
print("1")


total, unique_only_cnt, shared_cnt = shared_by_at_least_two(all_sequences)
print("RESULTS")

print("Total unique sequences:", total)
print("Unique to one sample:  ", unique_only_cnt)
print("Shared by ≥2 samples:  ", shared_cnt)
print("Percentage shared:     ", f"{shared_cnt/total*100:.2f}%")


DataFrame ERR629132 has 271894 lines.
DataFrame ERR629133 has 207891 lines.
DataFrame ERR629134 has 312985 lines.
DataFrame ERR629135 has 226513 lines.
DataFrame ERR629136 has 412538 lines.
DataFrame ERR629137 has 427962 lines.
DataFrame ERR629138 has 352423 lines.
DataFrame ERR629139 has 242008 lines.
total 2454214
1
Initial: seen=set(), unique_only=set()

After ERR629132:
After ERR629133:
After ERR629134:
After ERR629135:
After ERR629136:
After ERR629137:
After ERR629138:
After ERR629139:
RESULTS
Total unique sequences: 1154904
Unique to one sample:   645723
Shared by ≥2 samples:   509181
Percentage shared:      44.09%


In [5]:
#???
# define the base path for output csv files
#base_path = "../../../../2nd_6samples_output"
#base_path = "../../../../3rd_3samples_output"
#base_path = "../../../../4th_3samples_output"
#base_path = "../../../../6th_2samples_output"
base_path = "../../../../7th_6samples_output"
# base_path = "../../../../8th_2samples_output"
# base_path = "../../../../10th_2samples_output"
#base_path = "../../../../11th_3samples_output"
#base_path = "../../../../13th_3samples_output"

# names corresponding to the file names for bifi and ecoli
# sample_names = [
#     "MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001", "MEE-BIEVs-3_S9_L001_R1_001", 
#     "MEF-BIEVs-1_S1_L001_R1_001", "MEF-BIEVs-2_S5_L001_R1_001", "MEF-BIEVs-3_S6_L001_R1_001"
# ]
#dataset2
# sample_names = [
#     "SRR18078867", "SRR18078868", "SRR18078869", 
#     "SRR18078870", "SRR18078871", "SRR18078872"
# ]
#dataset3
# sample_names = [
#     "SRR20723831", "SRR20723832", "SRR20723833"
# ]
#dataset4
# sample_names = [
#     "SRR25896993", "SRR25896994", "SRR25896995"
# ]
#dataset6
# sample_names = [
#     "SRR12412943", "SRR12412944"
# ]
#dataset7
sample_names = [
    "DRR037104","DRR037108","DRR037112"
]
# #dataset8
# sample_names = [
#     "SRR1638859", "SRR1638860"
# ]
#dataset10
# sample_names = [
#     "SRR943111", "SRR943112"
# ]
# #dataset11
# sample_names = [
#     "SRR684065", "SRR684066","SRR684086"
# ]
#dataset13
# sample_names = [
#     "SRR8760721", "SRR8760722","SRR8760723"
# ]
# # bifi_names = ["MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001", "MEE-BIEVs-3_S9_L001_R1_001"]
# ecoli_names = [
#     "MEE-OMVs-1_S4_L001_R1_001", "MEE-OMVs-2_S3_L001_R1_001", "MEE-OMVs-3_S12_L001_R1_001", 
#     "MEF-OMVs-1_S2_L001_R1_001", "MEF-OMVs-2_S11_L001_R1_001", "MEF-OMVs-3_S10_L001_R1_001"
# ]

# columns to read from the original CSV files
read_columns = ['sequence', 'mirBase', 'linearfold', 'mirdeep2','qseqid_count']

df_all = initialize_dfs(sample_names)
df_all_with_match = initialize_dfs(sample_names)
# df_ecoli = initialize_dfs(ecoli_names)
# df_all = initialize_dfs(bifi_names + ecoli_names)

#df_bifi is a dictionary
read_csv_files(sample_names, df_all, base_path, read_columns)
filter_dataframes(sample_names, df_all, df_all_with_match)
total_num_rows = 0
for name in sample_names:
    num_rows = len(df_all_with_match[name])
    print(f"DataFrame {name} has {num_rows} lines.")
    total_num_rows += num_rows
print("total", total_num_rows)
all_sequences = {name: list(seq for seq in df['sequence']) for name, df in df_all_with_match.items()}
print("1")


total, unique_only_cnt, shared_cnt = shared_by_at_least_two(all_sequences)
print("RESULTS")

print("Total unique sequences:", total)
print("Unique to one sample:  ", unique_only_cnt)
print("Shared by ≥2 samples:  ", shared_cnt)
print("Percentage shared:     ", f"{shared_cnt/total*100:.2f}%")


DataFrame DRR037104 has 786255 lines.
DataFrame DRR037108 has 687780 lines.
DataFrame DRR037112 has 744932 lines.
total 2218967
1
Initial: seen=set(), unique_only=set()

After DRR037104:
After DRR037108:
After DRR037112:
RESULTS
Total unique sequences: 1759510
Unique to one sample:   1382489
Shared by ≥2 samples:   377021
Percentage shared:      21.43%


In [9]:

# define the base path for output csv files
#base_path = "../../../../2nd_6samples_output"
#base_path = "../../../../3rd_3samples_output"
#base_path = "../../../../4th_3samples_output"
#base_path = "../../../../6th_2samples_output"
#base_path = "../../../../7th_6samples_output"
base_path = "../../../../8th_2samples_output"
# base_path = "../../../../10th_2samples_output"
#base_path = "../../../../11th_3samples_output"
#base_path = "../../../../13th_3samples_output"

# names corresponding to the file names for bifi and ecoli
# sample_names = [
#     "MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001", "MEE-BIEVs-3_S9_L001_R1_001", 
#     "MEF-BIEVs-1_S1_L001_R1_001", "MEF-BIEVs-2_S5_L001_R1_001", "MEF-BIEVs-3_S6_L001_R1_001"
# ]
#dataset2
# sample_names = [
#     "SRR18078867", "SRR18078868", "SRR18078869", 
#     "SRR18078870", "SRR18078871", "SRR18078872"
# ]
#dataset3
# sample_names = [
#     "SRR20723831", "SRR20723832", "SRR20723833"
# ]
#dataset4
# sample_names = [
#     "SRR25896993", "SRR25896994", "SRR25896995"
# ]
#dataset6
# sample_names = [
#     "SRR12412943", "SRR12412944"
# ]
#dataset7
# sample_names = [
#     "DRR037104", "DRR037105","DRR037108","DRR037109","DRR037112","DRR037113"
# ]
# #dataset8
sample_names = [
    "SRR1638859", "SRR1638860"
]
#dataset10
# sample_names = [
#     "SRR943111", "SRR943112"
# ]
# #dataset11
# sample_names = [
#     "SRR684065", "SRR684066","SRR684086"
# ]
#dataset13
# sample_names = [
#     "SRR8760721", "SRR8760722","SRR8760723"
# ]
# # bifi_names = ["MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001", "MEE-BIEVs-3_S9_L001_R1_001"]
# ecoli_names = [
#     "MEE-OMVs-1_S4_L001_R1_001", "MEE-OMVs-2_S3_L001_R1_001", "MEE-OMVs-3_S12_L001_R1_001", 
#     "MEF-OMVs-1_S2_L001_R1_001", "MEF-OMVs-2_S11_L001_R1_001", "MEF-OMVs-3_S10_L001_R1_001"
# ]

# columns to read from the original CSV files
read_columns = ['sequence', 'mirBase', 'linearfold', 'mirdeep2','qseqid_count']

df_all = initialize_dfs(sample_names)
df_all_with_match = initialize_dfs(sample_names)
# df_ecoli = initialize_dfs(ecoli_names)
# df_all = initialize_dfs(bifi_names + ecoli_names)

#df_bifi is a dictionary
read_csv_files(sample_names, df_all, base_path, read_columns)
filter_dataframes(sample_names, df_all, df_all_with_match)
total_num_rows = 0
for name in sample_names:
    num_rows = len(df_all_with_match[name])
    print(f"DataFrame {name} has {num_rows} lines.")
    total_num_rows += num_rows
print("total", total_num_rows)
all_sequences = {name: list(seq for seq in df['sequence']) for name, df in df_all_with_match.items()}
print("1")


total, unique_only_cnt, shared_cnt = shared_by_at_least_two(all_sequences)
print("RESULTS")

print("Total unique sequences:", total)
print("Unique to one sample:  ", unique_only_cnt)
print("Shared by ≥2 samples:  ", shared_cnt)
print("Percentage shared:     ", f"{shared_cnt/total*100:.2f}%")


DataFrame SRR1638859 has 507293 lines.
DataFrame SRR1638860 has 241555 lines.
total 748848
1
Initial: seen=set(), unique_only=set()

After SRR1638859:
After SRR1638860:
RESULTS
Total unique sequences: 629486
Unique to one sample:   510124
Shared by ≥2 samples:   119362
Percentage shared:      18.96%


In [6]:

# define the base path for output csv files
#base_path = "../../../../2nd_6samples_output"
#base_path = "../../../../3rd_3samples_output"
#base_path = "../../../../4th_3samples_output"
#base_path = "../../../../6th_2samples_output"
#base_path = "../../../../7th_6samples_output"
# base_path = "../../../../8th_2samples_output"
base_path = "../../../../9th_12samples_output"
# base_path = "../../../../10th_2samples_output"
#base_path = "../../../../11th_3samples_output"
#base_path = "../../../../13th_3samples_output"

# names corresponding to the file names for bifi and ecoli
# sample_names = [
#     "MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001", "MEE-BIEVs-3_S9_L001_R1_001", 
#     "MEF-BIEVs-1_S1_L001_R1_001", "MEF-BIEVs-2_S5_L001_R1_001", "MEF-BIEVs-3_S6_L001_R1_001"
# ]
#dataset2
# sample_names = [
#     "SRR18078867", "SRR18078868", "SRR18078869", 
#     "SRR18078870", "SRR18078871", "SRR18078872"
# ]
#dataset3
# sample_names = [
#     "SRR20723831", "SRR20723832", "SRR20723833"
# ]
#dataset4
# sample_names = [
#     "SRR25896993", "SRR25896994", "SRR25896995"
# ]
#dataset6
# sample_names = [
#     "SRR12412943", "SRR12412944"
# ]
#dataset7
# sample_names = [
#     "DRR037104", "DRR037105","DRR037108","DRR037109","DRR037112","DRR037113"
# ]
# #dataset8
# sample_names = [
#     "SRR1638859", "SRR1638860"
# ]
# #dataset9
sample_names = [
    "SRR1295645", "SRR1295827","SRR1295840","SRR1295841","SRR1295843","SRR1295844",
    "SRR1295950","SRR1295974","SRR1295975","SRR1295978","SRR1295979"
]
#dataset10
# sample_names = [
#     "SRR943111", "SRR943112"
# ]
# #dataset11
# sample_names = [
#     "SRR684065", "SRR684066","SRR684086"
# ]
#dataset13
# sample_names = [
#     "SRR8760721", "SRR8760722","SRR8760723"
# ]
# # bifi_names = ["MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001", "MEE-BIEVs-3_S9_L001_R1_001"]
# ecoli_names = [
#     "MEE-OMVs-1_S4_L001_R1_001", "MEE-OMVs-2_S3_L001_R1_001", "MEE-OMVs-3_S12_L001_R1_001", 
#     "MEF-OMVs-1_S2_L001_R1_001", "MEF-OMVs-2_S11_L001_R1_001", "MEF-OMVs-3_S10_L001_R1_001"
# ]

# columns to read from the original CSV files
read_columns = ['sequence', 'mirBase', 'linearfold', 'mirdeep2','qseqid_count']

df_all = initialize_dfs(sample_names)
df_all_with_match = initialize_dfs(sample_names)
# df_ecoli = initialize_dfs(ecoli_names)
# df_all = initialize_dfs(bifi_names + ecoli_names)

#df_bifi is a dictionary
read_csv_files(sample_names, df_all, base_path, read_columns)
filter_dataframes(sample_names, df_all, df_all_with_match)
total_num_rows = 0
for name in sample_names:
    num_rows = len(df_all_with_match[name])
    print(f"DataFrame {name} has {num_rows} lines.")
    total_num_rows += num_rows
print("total", total_num_rows)
all_sequences = {name: list(seq for seq in df['sequence']) for name, df in df_all_with_match.items()}
print("1")


total, unique_only_cnt, shared_cnt = shared_by_at_least_two(all_sequences)
print("RESULTS")

print("Total unique sequences:", total)
print("Unique to one sample:  ", unique_only_cnt)
print("Shared by ≥2 samples:  ", shared_cnt)
print("Percentage shared:     ", f"{shared_cnt/total*100:.2f}%")


DataFrame SRR1295645 has 151935 lines.
DataFrame SRR1295827 has 28181 lines.
DataFrame SRR1295840 has 216373 lines.
DataFrame SRR1295841 has 43878 lines.
DataFrame SRR1295843 has 469490 lines.
DataFrame SRR1295844 has 3928 lines.
DataFrame SRR1295950 has 1833 lines.
DataFrame SRR1295974 has 77356 lines.
DataFrame SRR1295975 has 12595 lines.
DataFrame SRR1295978 has 378213 lines.
DataFrame SRR1295979 has 67599 lines.
total 1451381
1
Initial: seen=set(), unique_only=set()

After SRR1295645:
After SRR1295827:
After SRR1295840:
After SRR1295841:
After SRR1295843:
After SRR1295844:
After SRR1295950:
After SRR1295974:
After SRR1295975:
After SRR1295978:
After SRR1295979:
RESULTS
Total unique sequences: 1392991
Unique to one sample:   1338810
Shared by ≥2 samples:   54181
Percentage shared:      3.89%


In [11]:

# define the base path for output csv files
#base_path = "../../../../2nd_6samples_output"
#base_path = "../../../../3rd_3samples_output"
#base_path = "../../../../4th_3samples_output"
#base_path = "../../../../6th_2samples_output"
#base_path = "../../../../7th_6samples_output"
# base_path = "../../../../8th_2samples_output"
#base_path = "../../../../9th_12samples_output"
base_path = "../../../../10th_2samples_output"
#base_path = "../../../../11th_3samples_output"
#base_path = "../../../../13th_3samples_output"

# names corresponding to the file names for bifi and ecoli
# sample_names = [
#     "MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001", "MEE-BIEVs-3_S9_L001_R1_001", 
#     "MEF-BIEVs-1_S1_L001_R1_001", "MEF-BIEVs-2_S5_L001_R1_001", "MEF-BIEVs-3_S6_L001_R1_001"
# ]
#dataset2
# sample_names = [
#     "SRR18078867", "SRR18078868", "SRR18078869", 
#     "SRR18078870", "SRR18078871", "SRR18078872"
# ]
#dataset3
# sample_names = [
#     "SRR20723831", "SRR20723832", "SRR20723833"
# ]
#dataset4
# sample_names = [
#     "SRR25896993", "SRR25896994", "SRR25896995"
# ]
#dataset6
# sample_names = [
#     "SRR12412943", "SRR12412944"
# ]
#dataset7
# sample_names = [
#     "DRR037104", "DRR037105","DRR037108","DRR037109","DRR037112","DRR037113"
# ]
# #dataset8
# sample_names = [
#     "SRR1638859", "SRR1638860"
# ]
# #dataset9
# sample_names = [
#     "SRR1295645", "SRR1295827","SRR1295840","SRR1295841","SRR1295843","SRR1295844",
#     "SRR1295950","SRR1295974","SRR1295975","SRR1295977","SRR1295978","SRR1295979"
# ]
#dataset10
sample_names = [
    "SRR943111", "SRR943112"
]
# #dataset11
# sample_names = [
#     "SRR684065", "SRR684066","SRR684086"
# ]
#dataset13
# sample_names = [
#     "SRR8760721", "SRR8760722","SRR8760723"
# ]
# # bifi_names = ["MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001", "MEE-BIEVs-3_S9_L001_R1_001"]
# ecoli_names = [
#     "MEE-OMVs-1_S4_L001_R1_001", "MEE-OMVs-2_S3_L001_R1_001", "MEE-OMVs-3_S12_L001_R1_001", 
#     "MEF-OMVs-1_S2_L001_R1_001", "MEF-OMVs-2_S11_L001_R1_001", "MEF-OMVs-3_S10_L001_R1_001"
# ]

# columns to read from the original CSV files
read_columns = ['sequence', 'mirBase', 'linearfold', 'mirdeep2','qseqid_count']

df_all = initialize_dfs(sample_names)
df_all_with_match = initialize_dfs(sample_names)
# df_ecoli = initialize_dfs(ecoli_names)
# df_all = initialize_dfs(bifi_names + ecoli_names)

#df_bifi is a dictionary
read_csv_files(sample_names, df_all, base_path, read_columns)
filter_dataframes(sample_names, df_all, df_all_with_match)
total_num_rows = 0
for name in sample_names:
    num_rows = len(df_all_with_match[name])
    print(f"DataFrame {name} has {num_rows} lines.")
    total_num_rows += num_rows
print("total", total_num_rows)
all_sequences = {name: list(seq for seq in df['sequence']) for name, df in df_all_with_match.items()}
print("1")


total, unique_only_cnt, shared_cnt = shared_by_at_least_two(all_sequences)
print("RESULTS")

print("Total unique sequences:", total)
print("Unique to one sample:  ", unique_only_cnt)
print("Shared by ≥2 samples:  ", shared_cnt)
print("Percentage shared:     ", f"{shared_cnt/total*100:.2f}%")


DataFrame SRR943111 has 181148 lines.
DataFrame SRR943112 has 157054 lines.
total 338202
1
Initial: seen=set(), unique_only=set()

After SRR943111:
After SRR943112:
RESULTS
Total unique sequences: 250107
Unique to one sample:   162012
Shared by ≥2 samples:   88095
Percentage shared:      35.22%


In [12]:

# define the base path for output csv files
#base_path = "../../../../2nd_6samples_output"
#base_path = "../../../../3rd_3samples_output"
#base_path = "../../../../4th_3samples_output"
#base_path = "../../../../6th_2samples_output"
#base_path = "../../../../7th_6samples_output"
# base_path = "../../../../8th_2samples_output"
#base_path = "../../../../9th_12samples_output"
# base_path = "../../../../10th_2samples_output"
base_path = "../../../../11th_3samples_output"
#base_path = "../../../../13th_3samples_output"

# names corresponding to the file names for bifi and ecoli
# sample_names = [
#     "MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001", "MEE-BIEVs-3_S9_L001_R1_001", 
#     "MEF-BIEVs-1_S1_L001_R1_001", "MEF-BIEVs-2_S5_L001_R1_001", "MEF-BIEVs-3_S6_L001_R1_001"
# ]
#dataset2
# sample_names = [
#     "SRR18078867", "SRR18078868", "SRR18078869", 
#     "SRR18078870", "SRR18078871", "SRR18078872"
# ]
#dataset3
# sample_names = [
#     "SRR20723831", "SRR20723832", "SRR20723833"
# ]
#dataset4
# sample_names = [
#     "SRR25896993", "SRR25896994", "SRR25896995"
# ]
#dataset6
# sample_names = [
#     "SRR12412943", "SRR12412944"
# ]
#dataset7
# sample_names = [
#     "DRR037104", "DRR037105","DRR037108","DRR037109","DRR037112","DRR037113"
# ]
# #dataset8
# sample_names = [
#     "SRR1638859", "SRR1638860"
# ]
# #dataset9
# sample_names = [
#     "SRR1295645", "SRR1295827","SRR1295840","SRR1295841","SRR1295843","SRR1295844",
#     "SRR1295950","SRR1295974","SRR1295975","SRR1295977","SRR1295978","SRR1295979"
# ]
#dataset10
# sample_names = [
#     "SRR943111", "SRR943112"
# ]
# #dataset11
sample_names = [
    "SRR684065", "SRR684066","SRR684086"
]
#dataset13
# sample_names = [
#     "SRR8760721", "SRR8760722","SRR8760723"
# ]
# # bifi_names = ["MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001", "MEE-BIEVs-3_S9_L001_R1_001"]
# ecoli_names = [
#     "MEE-OMVs-1_S4_L001_R1_001", "MEE-OMVs-2_S3_L001_R1_001", "MEE-OMVs-3_S12_L001_R1_001", 
#     "MEF-OMVs-1_S2_L001_R1_001", "MEF-OMVs-2_S11_L001_R1_001", "MEF-OMVs-3_S10_L001_R1_001"
# ]

# columns to read from the original CSV files
read_columns = ['sequence', 'mirBase', 'linearfold', 'mirdeep2','qseqid_count']

df_all = initialize_dfs(sample_names)
df_all_with_match = initialize_dfs(sample_names)
# df_ecoli = initialize_dfs(ecoli_names)
# df_all = initialize_dfs(bifi_names + ecoli_names)

#df_bifi is a dictionary
read_csv_files(sample_names, df_all, base_path, read_columns)
filter_dataframes(sample_names, df_all, df_all_with_match)
total_num_rows = 0
for name in sample_names:
    num_rows = len(df_all_with_match[name])
    print(f"DataFrame {name} has {num_rows} lines.")
    total_num_rows += num_rows
print("total", total_num_rows)
all_sequences = {name: list(seq for seq in df['sequence']) for name, df in df_all_with_match.items()}
print("1")


total, unique_only_cnt, shared_cnt = shared_by_at_least_two(all_sequences)
print("RESULTS")

print("Total unique sequences:", total)
print("Unique to one sample:  ", unique_only_cnt)
print("Shared by ≥2 samples:  ", shared_cnt)
print("Percentage shared:     ", f"{shared_cnt/total*100:.2f}%")


DataFrame SRR684065 has 13151 lines.
DataFrame SRR684066 has 22004 lines.
DataFrame SRR684086 has 18043 lines.
total 53198
1
Initial: seen=set(), unique_only=set()

After SRR684065:
After SRR684066:
After SRR684086:
RESULTS
Total unique sequences: 43804
Unique to one sample:   36450
Shared by ≥2 samples:   7354
Percentage shared:      16.79%


In [13]:

# define the base path for output csv files
#base_path = "../../../../2nd_6samples_output"
#base_path = "../../../../3rd_3samples_output"
#base_path = "../../../../4th_3samples_output"
#base_path = "../../../../6th_2samples_output"
#base_path = "../../../../7th_6samples_output"
# base_path = "../../../../8th_2samples_output"
# base_path = "../../../../9th_12samples_output"
# base_path = "../../../../10th_2samples_output"
#base_path = "../../../../11th_3samples_output"
base_path = "../../../../12th_9samples_output"
#base_path = "../../../../13th_3samples_output"

# names corresponding to the file names for bifi and ecoli
# sample_names = [
#     "MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001", "MEE-BIEVs-3_S9_L001_R1_001", 
#     "MEF-BIEVs-1_S1_L001_R1_001", "MEF-BIEVs-2_S5_L001_R1_001", "MEF-BIEVs-3_S6_L001_R1_001"
# ]
#dataset2
# sample_names = [
#     "SRR18078867", "SRR18078868", "SRR18078869", 
#     "SRR18078870", "SRR18078871", "SRR18078872"
# ]
#dataset3
# sample_names = [
#     "SRR20723831", "SRR20723832", "SRR20723833"
# ]
#dataset4
# sample_names = [
#     "SRR25896993", "SRR25896994", "SRR25896995"
# ]
#dataset6
# sample_names = [
#     "SRR12412943", "SRR12412944"
# ]
#dataset7
# sample_names = [
#     "DRR037104", "DRR037105","DRR037108","DRR037109","DRR037112","DRR037113"
# ]
# #dataset8
# sample_names = [
#     "SRR1638859", "SRR1638860"
# ]
# #dataset9
# sample_names = [
#     "SRR1295645", "SRR1295827","SRR1295840","SRR1295841","SRR1295843","SRR1295844",
#     "SRR1295950","SRR1295974","SRR1295975","SRR1295977","SRR1295978","SRR1295979"
# ]
#dataset10
# sample_names = [
#     "SRR943111", "SRR943112"
# ]
# #dataset11
# sample_names = [
#     "SRR684065", "SRR684066","SRR684086"
# ]
# #dataset12
sample_names = [
    "SRR9851810", "SRR9851811","SRR9851812","SRR9851813","SRR9851814","SRR9851815",
    "SRR9851816","SRR9851817","SRR9851818"

]
#dataset13
# sample_names = [
#     "SRR8760721", "SRR8760722","SRR8760723"
# ]
# # bifi_names = ["MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001", "MEE-BIEVs-3_S9_L001_R1_001"]
# ecoli_names = [
#     "MEE-OMVs-1_S4_L001_R1_001", "MEE-OMVs-2_S3_L001_R1_001", "MEE-OMVs-3_S12_L001_R1_001", 
#     "MEF-OMVs-1_S2_L001_R1_001", "MEF-OMVs-2_S11_L001_R1_001", "MEF-OMVs-3_S10_L001_R1_001"
# ]

# columns to read from the original CSV files
read_columns = ['sequence', 'mirBase', 'linearfold', 'mirdeep2','qseqid_count']

df_all = initialize_dfs(sample_names)
df_all_with_match = initialize_dfs(sample_names)
# df_ecoli = initialize_dfs(ecoli_names)
# df_all = initialize_dfs(bifi_names + ecoli_names)

#df_bifi is a dictionary
read_csv_files(sample_names, df_all, base_path, read_columns)
filter_dataframes(sample_names, df_all, df_all_with_match)
total_num_rows = 0
for name in sample_names:
    num_rows = len(df_all_with_match[name])
    print(f"DataFrame {name} has {num_rows} lines.")
    total_num_rows += num_rows
print("total", total_num_rows)
all_sequences = {name: list(seq for seq in df['sequence']) for name, df in df_all_with_match.items()}
print("1")


total, unique_only_cnt, shared_cnt = shared_by_at_least_two(all_sequences)
print("RESULTS")

print("Total unique sequences:", total)
print("Unique to one sample:  ", unique_only_cnt)
print("Shared by ≥2 samples:  ", shared_cnt)
print("Percentage shared:     ", f"{shared_cnt/total*100:.2f}%")


DataFrame SRR9851810 has 322 lines.
DataFrame SRR9851811 has 419 lines.
DataFrame SRR9851812 has 411 lines.
DataFrame SRR9851813 has 332 lines.
DataFrame SRR9851814 has 226 lines.
DataFrame SRR9851815 has 399 lines.
DataFrame SRR9851816 has 326 lines.
DataFrame SRR9851817 has 322 lines.
DataFrame SRR9851818 has 373 lines.
total 3130
1
Initial: seen=set(), unique_only=set()

After SRR9851810:
After SRR9851811:
After SRR9851812:
After SRR9851813:
After SRR9851814:
After SRR9851815:
After SRR9851816:
After SRR9851817:
After SRR9851818:
RESULTS
Total unique sequences: 2550
Unique to one sample:   2300
Shared by ≥2 samples:   250
Percentage shared:      9.80%


In [14]:

# define the base path for output csv files
#base_path = "../../../../2nd_6samples_output"
#base_path = "../../../../3rd_3samples_output"
#base_path = "../../../../4th_3samples_output"
#base_path = "../../../../6th_2samples_output"
#base_path = "../../../../7th_6samples_output"
# base_path = "../../../../8th_2samples_output"
# base_path = "../../../../9th_12samples_output"
# base_path = "../../../../10th_2samples_output"
#base_path = "../../../../11th_3samples_output"
# base_path = "../../../../12th_9samples_output"
base_path = "../../../../13th_3samples_output"

# names corresponding to the file names for bifi and ecoli
# sample_names = [
#     "MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001", "MEE-BIEVs-3_S9_L001_R1_001", 
#     "MEF-BIEVs-1_S1_L001_R1_001", "MEF-BIEVs-2_S5_L001_R1_001", "MEF-BIEVs-3_S6_L001_R1_001"
# ]
#dataset2
# sample_names = [
#     "SRR18078867", "SRR18078868", "SRR18078869", 
#     "SRR18078870", "SRR18078871", "SRR18078872"
# ]
#dataset3
# sample_names = [
#     "SRR20723831", "SRR20723832", "SRR20723833"
# ]
#dataset4
# sample_names = [
#     "SRR25896993", "SRR25896994", "SRR25896995"
# ]
#dataset6
# sample_names = [
#     "SRR12412943", "SRR12412944"
# ]
#dataset7
# sample_names = [
#     "DRR037104", "DRR037105","DRR037108","DRR037109","DRR037112","DRR037113"
# ]
# #dataset8
# sample_names = [
#     "SRR1638859", "SRR1638860"
# ]
# #dataset9
# sample_names = [
#     "SRR1295645", "SRR1295827","SRR1295840","SRR1295841","SRR1295843","SRR1295844",
#     "SRR1295950","SRR1295974","SRR1295975","SRR1295977","SRR1295978","SRR1295979"
# ]
#dataset10
# sample_names = [
#     "SRR943111", "SRR943112"
# ]
# #dataset11
# sample_names = [
#     "SRR684065", "SRR684066","SRR684086"
# ]
# #dataset12
# sample_names = [
#     "SRR9851810", "SRR9851811","SRR9851812","SRR9851813","SRR9851814","SRR9851815",
#     "SRR9851816","SRR9851817","SRR9851818"

# ]
#dataset13
sample_names = [
    "SRR8760721", "SRR8760722","SRR8760723"
]
# # bifi_names = ["MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001", "MEE-BIEVs-3_S9_L001_R1_001"]
# ecoli_names = [
#     "MEE-OMVs-1_S4_L001_R1_001", "MEE-OMVs-2_S3_L001_R1_001", "MEE-OMVs-3_S12_L001_R1_001", 
#     "MEF-OMVs-1_S2_L001_R1_001", "MEF-OMVs-2_S11_L001_R1_001", "MEF-OMVs-3_S10_L001_R1_001"
# ]

# columns to read from the original CSV files
read_columns = ['sequence', 'mirBase', 'linearfold', 'mirdeep2','qseqid_count']

df_all = initialize_dfs(sample_names)
df_all_with_match = initialize_dfs(sample_names)
# df_ecoli = initialize_dfs(ecoli_names)
# df_all = initialize_dfs(bifi_names + ecoli_names)

#df_bifi is a dictionary
read_csv_files(sample_names, df_all, base_path, read_columns)
filter_dataframes(sample_names, df_all, df_all_with_match)
total_num_rows = 0
for name in sample_names:
    num_rows = len(df_all_with_match[name])
    print(f"DataFrame {name} has {num_rows} lines.")
    total_num_rows += num_rows
print("total", total_num_rows)
all_sequences = {name: list(seq for seq in df['sequence']) for name, df in df_all_with_match.items()}
print("1")


total, unique_only_cnt, shared_cnt = shared_by_at_least_two(all_sequences)
print("RESULTS")

print("Total unique sequences:", total)
print("Unique to one sample:  ", unique_only_cnt)
print("Shared by ≥2 samples:  ", shared_cnt)
print("Percentage shared:     ", f"{shared_cnt/total*100:.2f}%")


DataFrame SRR8760721 has 1140 lines.
DataFrame SRR8760722 has 2773 lines.
DataFrame SRR8760723 has 3584 lines.
total 7497
1
Initial: seen=set(), unique_only=set()

After SRR8760721:
After SRR8760722:
After SRR8760723:
RESULTS
Total unique sequences: 6795
Unique to one sample:   6250
Shared by ≥2 samples:   545
Percentage shared:      8.02%


In [16]:
import os
import pandas as pd
from itertools import combinations
from concurrent.futures import ThreadPoolExecutor

base_path = "../../../../14th_18samples_output"

# names corresponding to the file names for bifi and ecoli
# sample_names = [
#     "MEE-BIEVs-1_S7_L001_R1_001", "MEE-BIEVs-2_S8_L001_R1_001", "MEE-BIEVs-3_S9_L001_R1_001", 
#     "MEF-BIEVs-1_S1_L001_R1_001", "MEF-BIEVs-2_S5_L001_R1_001", "MEF-BIEVs-3_S6_L001_R1_001"
# ]
#dataset2
# sample_names = [
#     "SRR18078867", "SRR18078868", "SRR18078869", 
#     "SRR18078870", "SRR18078871", "SRR18078872"
# ]
#dataset3
# sample_names = [
#     "SRR20723831", "SRR20723832", "SRR20723833"
# ]
#dataset4
# sample_names = [
#     "SRR25896993", "SRR25896994", "SRR25896995"
# ]
#dataset6
# sample_names = [
#     "SRR12412943", "SRR12412944"
# ]
#dataset7
# sample_names = [
#     "DRR037104", "DRR037105","DRR037108","DRR037109","DRR037112","DRR037113"
# ]
# #dataset8
# sample_names = [
#     "SRR1638859", "SRR1638860"
# ]
# #dataset9
# sample_names = [
#     "SRR1295645", "SRR1295827","SRR1295840","SRR1295841","SRR1295843","SRR1295844",
#     "SRR1295950","SRR1295974","SRR1295975","SRR1295977","SRR1295978","SRR1295979"
# ]
#dataset10
# sample_names = [
#     "SRR943111", "SRR943112"
# ]
# #dataset11
# sample_names = [
#     "SRR684065", "SRR684066","SRR684086"
# ]
# #dataset12
# sample_names = [
#     "SRR9851810", "SRR9851811","SRR9851812","SRR9851813","SRR9851814","SRR9851815",
#     "SRR9851816","SRR9851817","SRR9851818"

# ]
#dataset13
# sample_names = [
#     "SRR8760721", "SRR8760722","SRR8760723"
# ]
#dataset14
sample_names = [
    "SRR3382456", "SRR3382457","SRR3382807","SRR3382808","SRR3382809","SRR3383247",
    "SRR3383248","SRR3383249","SRR3383250","SRR3383251","SRR3383252","SRR3383253",
    "SRR3383254","SRR3383255","SRR3383256","SRR3383257","SRR3383258","SRR3383259"
]
def find_common_sequences_fast(samples):
    """
    Fast, correct version of find_common_sequences().

    Parameters
    ----------
    samples : dict
        key = sample name
        value = list or set of sequences

    Returns
    -------
    combos : list of list of sample names
    counts : list of int (intersection counts)
    """

    sample_names = list(samples.keys())

    # ---- Step 1. Collect all unique sequences
    all_sequences = pd.Index(np.unique(np.concatenate(list(samples.values()))))

    # ---- Step 2. Build boolean matrix: rows = sequences, columns = samples
    presence = np.zeros((len(all_sequences), len(sample_names)), dtype=bool)
    seq_index = {seq: i for i, seq in enumerate(all_sequences)}

    for j, name in enumerate(sample_names):
        seqs = samples[name]
        idx = [seq_index[s] for s in seqs if s in seq_index]
        presence[idx, j] = True

    # ---- Step 3. Vectorized intersection counts for all combinations
    combos = []
    counts = []

    # precompute transposed matrix for speed
    P = presence.T

    for k in range(1, len(sample_names) + 1):
        for combo in combinations(range(len(sample_names)), k):
            # Logical AND across these samples
            intersect_mask = np.logical_and.reduce(P[list(combo)], axis=0)
            count = np.count_nonzero(intersect_mask)
            combos.append([sample_names[i] for i in combo])
            counts.append(int(count))

    return combos, counts


read_columns = ['sequence', 'mirBase', 'linearfold', 'mirdeep2','qseqid_count']

df_all = initialize_dfs(sample_names)
df_all_with_match = initialize_dfs(sample_names)
# df_ecoli = initialize_dfs(ecoli_names)
# df_all = initialize_dfs(bifi_names + ecoli_names)

#df_bifi is a dictionary
read_csv_files(sample_names, df_all, base_path, read_columns)
filter_dataframes(sample_names, df_all, df_all_with_match)
total_num_rows = 0
for name in sample_names:
    num_rows = len(df_all_with_match[name])
    print(f"DataFrame {name} has {num_rows} lines.")
    total_num_rows += num_rows
print("total", total_num_rows)
all_sequences = {name: list(seq for seq in df['sequence']) for name, df in df_all_with_match.items()}
print("1")


total, unique_only_cnt, shared_cnt = shared_by_at_least_two(all_sequences)
print("RESULTS")

print("Total unique sequences:", total)
print("Unique to one sample:  ", unique_only_cnt)
print("Shared by ≥2 samples:  ", shared_cnt)
print("Percentage shared:     ", f"{shared_cnt/total*100:.2f}%")


DataFrame SRR3382456 has 9930 lines.
DataFrame SRR3382457 has 15845 lines.
DataFrame SRR3382807 has 6559 lines.
DataFrame SRR3382808 has 15917 lines.
DataFrame SRR3382809 has 7325 lines.
DataFrame SRR3383247 has 10021 lines.
DataFrame SRR3383248 has 20552 lines.
DataFrame SRR3383249 has 13466 lines.
DataFrame SRR3383250 has 11837 lines.
DataFrame SRR3383251 has 11926 lines.
DataFrame SRR3383252 has 19725 lines.
DataFrame SRR3383253 has 12925 lines.
DataFrame SRR3383254 has 11454 lines.
DataFrame SRR3383255 has 18659 lines.
DataFrame SRR3383256 has 22032 lines.
DataFrame SRR3383257 has 12235 lines.
DataFrame SRR3383258 has 46324 lines.
DataFrame SRR3383259 has 12550 lines.
total 279282
1
Initial: seen=set(), unique_only=set()

After SRR3382456:
After SRR3382457:
After SRR3382807:
After SRR3382808:
After SRR3382809:
After SRR3383247:
After SRR3383248:
After SRR3383249:
After SRR3383250:
After SRR3383251:
After SRR3383252:
After SRR3383253:
After SRR3383254:
After SRR3383255:
After SRR338

In [17]:
import os
import pandas as pd
from itertools import combinations
from concurrent.futures import ThreadPoolExecutor
from collections import Counter
# initialize dataFrames
def initialize_dfs(names):
    return {name: pd.DataFrame() for name in names}
    
# function to read csv files into dataframes
def read_csv_files(names, df_dict, base_path, columns):
    for name in names:
        file_path = os.path.join(base_path, name, "produce_final_form/final_form.csv")
        df_dict[name] = pd.read_csv(file_path, sep=',', usecols=columns)


# Generate the final list format
def format_memberships(sample_sequences, result):
    formatted = []
    data = []
    
# Filter conditions: mirBase, linearfold, mirdeep2
def filter_dataframes(names, df_source, df_with_match):
    for name in names:
        df_with_match[name] = filter_any_condition_met(df_source[name])
        # df_all_zero[name] = filter_none_met(df_source[name])

def filter_any_condition_met(df):
    filtered_df = df[(df[['mirBase', 'linearfold', 'mirdeep2']].sum(axis=1) > 0)]
    filtered_df = filtered_df.loc[:, ['sequence', 'qseqid_count']]
    # print("filtered_df",filtered_df)
    return filtered_df

def shared_by_at_least_two(all_sequences):
    all_sets = {k: set(v) for k, v in all_sequences.items()}

    total_set = set.union(*all_sets.values())  # union
    unique_only = set()
    seen = set()

    print(f"Initial: seen={seen}, unique_only={unique_only}\n")

    for name, s in all_sets.items():
        print(f"After {name}:")
        duplicates = seen & s
        unique_only |= (s - seen)
        unique_only -= duplicates
        seen |= s
        # print(f"After {name}:")
        # print(f"  sample={s}")
        # print(f"  duplicates={duplicates}")
        # print(f"  seen={seen}")
        # print(f"  unique_only={unique_only}\n")

    shared = total_set - unique_only
    return len(total_set), len(unique_only), len(shared)

base_path = "../../../../15th_20samples_output"


# #dataset15
sample_names = [
    "ERR1296699", "ERR1296700","ERR1296701","ERR1296702","ERR1296703","ERR1296704",
    "ERR1396754","ERR1396755","ERR1396756","ERR1396757","ERR1396758","ERR1396759",
    "ERR1396760","ERR1396761","ERR1396762","ERR1396763","ERR1396764","ERR1396765",
    "ERR1396766","ERR1396767"

]

read_columns = ['sequence', 'mirBase', 'linearfold', 'mirdeep2','qseqid_count']

df_all = initialize_dfs(sample_names)
df_all_with_match = initialize_dfs(sample_names)
# df_ecoli = initialize_dfs(ecoli_names)
# df_all = initialize_dfs(bifi_names + ecoli_names)

#df_bifi is a dictionary
read_csv_files(sample_names, df_all, base_path, read_columns)
filter_dataframes(sample_names, df_all, df_all_with_match)
# print(df_all_with_match)
total_num_rows = 0
for name in sample_names:
    num_rows = len(df_all_with_match[name])
    print(f"DataFrame {name} has {num_rows} lines.")
    total_num_rows += num_rows
print("total", total_num_rows)
all_sequences = {name: list(seq for seq in df['sequence']) for name, df in df_all_with_match.items()}
print("1")


total, unique_only_cnt, shared_cnt = shared_by_at_least_two(all_sequences)
print("RESULTS")

print("Total unique sequences:", total)
print("Unique to one sample:  ", unique_only_cnt)
print("Shared by ≥2 samples:  ", shared_cnt)
print("Percentage shared:     ", f"{shared_cnt/total*100:.2f}%")


DataFrame ERR1296699 has 612013 lines.
DataFrame ERR1296700 has 917973 lines.
DataFrame ERR1296701 has 655823 lines.
DataFrame ERR1296702 has 616731 lines.
DataFrame ERR1296703 has 921099 lines.
DataFrame ERR1296704 has 659552 lines.
DataFrame ERR1396754 has 479818 lines.
DataFrame ERR1396755 has 319400 lines.
DataFrame ERR1396756 has 137916 lines.
DataFrame ERR1396757 has 59246 lines.
DataFrame ERR1396758 has 503416 lines.
DataFrame ERR1396759 has 505179 lines.
DataFrame ERR1396760 has 279953 lines.
DataFrame ERR1396761 has 480896 lines.
DataFrame ERR1396762 has 320843 lines.
DataFrame ERR1396763 has 139167 lines.
DataFrame ERR1396764 has 59340 lines.
DataFrame ERR1396765 has 505195 lines.
DataFrame ERR1396766 has 507268 lines.
DataFrame ERR1396767 has 281355 lines.
total 8962183
1
Initial: seen=set(), unique_only=set()

After ERR1296699:
After ERR1296700:
After ERR1296701:
After ERR1296702:
After ERR1296703:
After ERR1296704:
After ERR1396754:
After ERR1396755:
After ERR1396756:
Afte